In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from pandas.api.types import is_numeric_dtype
import statsmodels
from statsmodels.stats.multitest import multipletests

In [ ]:
# Function to remove clusters containing < 10% patients
# output is also list with top x outlier patients (patients most typically found in small clusters)
def remove_small_clusters(df: pd.DataFrame, n_outliers: int, verbose: bool):
    valid_results = df[df['relative_cluster_sizes'].apply(lambda d: all(value >= 0.1 for value in d.values()))]
    outlier_results = df[df['relative_cluster_sizes'].apply(lambda f: any(value < 0.1 for value in f.values()))]
    outlier_patients = {}
    for index, row in outlier_results.iterrows():
        cluster_sizes = row['relative_cluster_sizes']
        small_clusters = [cluster for cluster, size in cluster_sizes.items() if size < 0.1]
        patients = row['y_pred_idx']
        clusters = row['y_pred']
        for patient, cluster in zip(patients, clusters):
            if cluster in small_clusters:
                if patient not in outlier_patients:
                    outlier_patients[patient] = 1
                else:
                    outlier_patients[patient] += 1
    for key in outlier_patients:
        outlier_patients[key] /= len(outlier_results)
    top_outliers = sorted(outlier_patients.items(), key=lambda x: x[1], reverse=True)[:n_outliers]
    df_top_outliers = pd.DataFrame(data=top_outliers, columns=['Patient ID', 'Count'])
    top_outlier_patients = [patient for patient, count in top_outliers]
    if verbose == True:
        print(f"Top {n_outliers} outlier patients: {top_outlier_patients}")
    return valid_results, outlier_results, df_top_outliers

In [ ]:
# Function to perform clinical enrichment of labels to each row
def clinical_enrichment(results, clinical_data):
    # Sort labels
    results["sorted_y_pred_idx"] = results["y_pred_idx"].apply(sorted)
    results["sorted_y_pred"] = results.apply(
        lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]],
        axis=1)
    
    # Add clinical data
    clinical_data = clinical_data[['Patient ID', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                   'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                   'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                   'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                   'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                   'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade', 'TMB (nonsynonymous)',
                                   'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                   'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                   'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
    clinical_data.set_index('Patient ID', inplace=True)
    
    # Convert necessary data 
    clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
    clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
    clinical_data_columns = [col for col in clinical_data.columns if col != 'Patient ID']

    def get_filtered_clinical_data(patient_ids, clinical_data, column_name):
        filtered_data = clinical_data.loc[patient_ids]
        return filtered_data[column_name].values.tolist()
    for column in clinical_data_columns:
        results[column] = results.apply(lambda row: get_filtered_clinical_data(row['sorted_y_pred_idx'], clinical_data, column), axis=1)
    
    # Logrank test
    def calculate_logrank_pvalue(row):
        df = pd.DataFrame({
            'cluster': row['sorted_y_pred'],
            'vital_status': row['Overall Survival Status'],
            'days_to_death': row['Overall Survival (Months)']
        })
        kmf = KaplanMeierFitter()
        test_results = multivariate_logrank_test(df['days_to_death'], df['cluster'], df['vital_status'])
        return test_results.p_value
    results['pvalue_logrank'] = results.apply(calculate_logrank_pvalue, axis=1)
    
    # Function for p-values (Kruskal-Wallis, chi2)
    def pvalue_tests(row, clinical_data_columns):
        pvalues = []
        for variable in clinical_data_columns:
            df = pd.DataFrame({
                'cluster': row['y_pred'],
                variable: row[variable]
            })
            if pd.api.types.is_numeric_dtype(df[variable]):
                # Kruskal-Wallis test for numerical variables
                test_numerical = [df[df['cluster'] == cluster][variable].dropna().to_numpy() for cluster in df['cluster'].unique()]
                stat, p_value_kruskal = kruskal(*test_numerical)
                pvalues.append(p_value_kruskal)
            else:
                # Chi-square contingency test for categorical variables
                test_discrete = pd.crosstab(df['cluster'], df[variable])
                chi2, p_value_chi2, dof, freq = chi2_contingency(test_discrete)
                pvalues.append(p_value_chi2)
        reject, pvals_corr, asidack, abonf = multipletests(pvals=pvalues, alpha=0.05, method='fdr_bh')
        for idx, variable in enumerate(clinical_data_columns):
            row[f"pvalue_{variable}"] = pvals_corr[idx]
        return row
    
    clinical_data_columns = [col for col in clinical_data_columns if col not in ['Overall Survival Status', 'Overall Survival (Months)']]
    results = results.apply(lambda row: pvalue_tests(row, clinical_data_columns), axis=1)
    columns_to_check = [f'pvalue_{variable}' for variable in clinical_data_columns]
    results['n_enriched_clinical'] = (results[columns_to_check] < 0.05).sum(axis=1)
    return results

In [ ]:
# Function to calculate stability metrics
def calculate_stability_metrics(results: pd.DataFrame, random_state=None, progress_bar=True):

    alg_stability = results[['dataset', 'view_combination', 'algorithm', 'n_clusters', 'missing_percentage', 'amputation_mechanism', 'imputation', 'run_n', "sorted_y_pred", 
                             "sorted_y_pred_idx", 'silhouette', 'normalised_silhouette', 'vrc', 'db', 'dbcv', 'dunn', "dhi", "ssei", 'rsi', 'bhi', 'pvalue_logrank', 'n_enriched_clinical']]
    if alg_stability["imputation"].nunique() != 1:
        alg_stability = alg_stability.loc[
            (alg_stability["missing_percentage"] == 0) | (alg_stability["imputation"])
            ]
    alg_uns_metrics = alg_stability.drop(columns=["sorted_y_pred", "sorted_y_pred_idx",'imputation', 'run_n'])
    
    # Group by taking mean of metrics
    alg_uns_metrics = alg_uns_metrics.groupby(
        ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"], as_index=False).mean()

    iterator = alg_stability["dataset"].unique()
    if progress_bar:
        iterator = tqdm(iterator)

    for dataset in iterator:
        preds_dataset = alg_stability.loc[
            (alg_stability["dataset"] == dataset), ["missing_percentage", "algorithm", 'amputation_mechanism', 'n_clusters', 
                                                    'view_combination', "run_n", "sorted_y_pred", "sorted_y_pred_idx"]]
        for alg in preds_dataset["algorithm"].unique():
            pred_alg = preds_dataset[preds_dataset["algorithm"] == alg]
            for missing_percentage in pred_alg["missing_percentage"].unique():
                pred_missing_alg = pred_alg[pred_alg["missing_percentage"] == missing_percentage]
                for amputation_mechanism in pred_missing_alg["amputation_mechanism"].unique():
                    pred_missing_ampt_alg = pred_missing_alg[
                        pred_missing_alg["amputation_mechanism"] == amputation_mechanism]
                    
                    for view in pred_missing_ampt_alg["view_combination"].unique():
                        pred_missing_ampt_alg_view = pred_missing_ampt_alg[
                            pred_missing_ampt_alg["view_combination"] == view]
                        for cluster in pred_missing_ampt_alg_view["n_clusters"].unique():
                            pred_missing_ampt_alg_view_clus = pred_missing_ampt_alg_view[
                                pred_missing_ampt_alg_view['n_clusters'] == cluster]

                            amis, aris = [], []
                            
                            for run_1, run_2 in set(itertools.combinations(pred_missing_ampt_alg_view_clus["run_n"].unique(), 2)):
                                pred1_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                                pred2_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                                
                                pred1_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                                pred2_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
        
                                # Only select samples in common for stability metrics
                                common_samples = list(set(pred1_idx) & set(pred2_idx))
                                pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                                pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
        
                                amis.append(adjusted_mutual_info_score(pred1_common, pred2_common)), aris.append(
                                    adjusted_rand_score(pred1_common, pred2_common))
        
                            alg_uns_metrics.loc[(alg_uns_metrics["dataset"] == dataset) &
                                                (alg_uns_metrics["missing_percentage"] == missing_percentage) &
                                                (alg_uns_metrics["amputation_mechanism"] == amputation_mechanism) &
                                                (alg_uns_metrics["algorithm"] == alg) & 
                                                (alg_uns_metrics["view_combination"] == view) & 
                                                (alg_uns_metrics["n_clusters"] == cluster),
                            ["AMI", "ARI"]] = [np.mean(amis), np.mean(aris)]
    return alg_uns_metrics

In [ ]:
# Function to normalise metrics with respect to a variable
def add_normalised_metric(df, variable_to_normalise, metric, greater_is_better=True):
    possible_variables = ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            relative_score = scores / scores.max()
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df[key] == value)
            df.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df[f'normalised_{metric}'] = float('nan')
    recursive_loop(df, valid_variables, {})
    return df

# Second benchmarking: obtaining the best algorithm

In [ ]:
results1_file = pd.read_csv('benchmarking_files/first_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval, 
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
results2_file = pd.read_csv('benchmarking_files/second_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval,
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
view_combination = results2_file['view_combination'].iloc[0]
results1_combination = results1_file[(results1_file['n_clusters'] == 2) & (results1_file['view_combination'] == view_combination)]
results2_df = pd.concat([results1_combination, results2_file])

def substitute_ampmech(df):
    amputation_mechanisms = ['edm', 'pm', 'mnar', 'mcar']
    no_amputation_df = df[df['amputation_mechanism'] == 'No']
    new_rows_list = []
    for mechanism in amputation_mechanisms:
        temp_df = no_amputation_df.copy()
        temp_df['amputation_mechanism'] = mechanism
        new_rows_list.append(temp_df)
    new_amputation_rows = pd.concat(new_rows_list, ignore_index=True)
    df_no_no = df[df['amputation_mechanism'] != 'No']
    df_updated = pd.concat([df_no_no, new_amputation_rows], ignore_index=True)
    return df_updated

results2_df = substitute_ampmech(results2_df)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
valid_results2, outlier_results2, outlier_patients2 = remove_small_clusters(results2_df, 20, verbose=False)
results_clin2 = clinical_enrichment(valid_results2, clinical_data_file)
normalised_silhouette2 = add_normalised_metric(results_clin2, metric='silhouette', variable_to_normalise='missing_percentage', greater_is_better=True)
stability_metrics_results2 = calculate_stability_metrics(normalised_silhouette2, random_state=42, progress_bar=True)
normalised_ami2 = add_normalised_metric(stability_metrics_results2, metric='AMI', variable_to_normalise='missing_percentage', greater_is_better=True)
results2 = normalised_ami2.copy()
results2['combined_metric'] = results2[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results2 = results2.sort_values('combined_metric', ascending=False)
results2

### Pointplots

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    ax.grid(alpha=0.3)
    plt.tight_layout()

fig, ax = plt.subplots(2, 3, figsize=(12,6))
# Pointplots by algorithm
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0, 0], 'AMI + silhouette score')
plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[0, 1], 'Log-rank test p-value')
plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[0,2], 'No. enriched clinical parameters')
# Pointplots by amputation mechanism
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[1, 0], 'AMI + silhouette score')
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[1, 1], 'Log-rank test p-value')
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[1,2], 'No. enriched clinical parameters')

# single legend at end of row
handles_alg, labels_alg = ax[0, 0].get_legend_handles_labels()
handles_amp, labels_amp = ax[1, 0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[0, 2].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
ax[1, 2].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')


fig.subplots_adjust(wspace = 0.3, hspace = 0.35)
plt.savefig('figures/pointplots_bench2.svg', bbox_inches='tight')
plt.show()

In [ ]:
def plot_pointplots(df, colname_var, colname_metric, ax, ylabel):
    sns.lineplot(x='missing_percentage', y=colname_metric, data=df, hue=colname_var, style=colname_var,
                  palette="colorblind", ax=ax, markers=True, dashes=True, ci=None)
    ax.set_xlabel('Missing percentage')
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=-0.05, top=(max(df[colname_metric])+0.05 if max(df[colname_metric])+0.05 > 1.1  else 1.1))
    ax.grid(alpha=0.3)
    plt.tight_layout()

fig, ax = plt.subplots(2, 5, figsize=(18,8))
# Pointplots by algorithm
plot_pointplots(results2, 'algorithm', 'combined_metric', ax[0, 0], 'AMI + silhouette score')
plot_pointplots(results2, 'algorithm', 'normalised_silhouette', ax[0, 1], 'Silhouette score')
plot_pointplots(results2, 'algorithm', 'normalised_AMI', ax[0, 2], 'AMI score')
plot_pointplots(results2, 'algorithm', 'pvalue_logrank', ax[0, 3], 'Log-rank test p-value')
plot_pointplots(results2, 'algorithm', 'n_enriched_clinical', ax[0, 4], 'Number of enriched clinical parameters')
# Pointplots by amputation mechanism
plot_pointplots(results2, 'amputation_mechanism', 'combined_metric', ax[1, 0], 'AMI + silhouette score')
plot_pointplots(results2, 'amputation_mechanism', 'normalised_silhouette', ax[1, 1], 'Silhouette score')
plot_pointplots(results2, 'amputation_mechanism', 'normalised_AMI', ax[1, 2], 'AMI score')
plot_pointplots(results2, 'amputation_mechanism', 'pvalue_logrank', ax[1, 3], 'Log-rank test p-value')
plot_pointplots(results2, 'amputation_mechanism', 'n_enriched_clinical', ax[1, 4], 'Number of enriched clinical parameters')

# single legend at end of row
handles_alg, labels_alg = ax[0, 0].get_legend_handles_labels()
handles_amp, labels_amp = ax[1, 0].get_legend_handles_labels()
for i in ax.flat:
    i.legend().remove()
ax[0, 4].legend(handles_alg, labels_alg, title="Algorithm", loc='upper right')
ax[1, 4].legend(handles_amp, labels_amp, title="Amputation\nmechanism", loc='upper right')

fig.subplots_adjust(wspace = 0.3, hspace = 0.35)
plt.savefig('figures/pointplots_bench2comp.svg', bbox_inches='tight')
plt.show()